# TP1 — Le réseau du métro de Montréal
## Carnet, graphes, Dijkstra — et un premier modèle

**Nom :** Matthew_Barreau &nbsp;&nbsp;&nbsp; **Coéquipier :** _s'il y a lieu_ &nbsp;&nbsp;&nbsp; **Groupe :** gr. 07979

---

Ce TP se fait **seul ou en équipe de 2**. Une équipe remet **un seul carnet** et
indique à la fin qui a fait quoi.

**Chaque question, y compris les questions de code, se termine par une cellule
d'observation en Markdown.** Écrivez-y ce que vous constatez, en phrases complètes,
en français. Une cellule d'observation laissée telle quelle vaut 0 pour la question.

Avant de remettre : **Kernel → Redémarrer et tout exécuter**.

## Partie 2 — Charger et explorer le réseau

Le fichier `donnees/metro_montreal.csv` contient une ligne par lien entre deux
stations voisines, avec la durée approximative en minutes. Le fichier
`donnees/stations.csv` donne la ligne de chaque station.

In [ ]:
import csv

reseau = {}

with open("donnees/metro_montreal.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        a = ligne["station_a"]
        b = ligne["station_b"]
        minutes = int(ligne["minutes"])
        reseau.setdefault(a, []).append((b, minutes))
        reseau.setdefault(b, []).append((a, minutes))

infos_station = {}
with open("donnees/stations.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        infos_station[ligne["station"]] = ligne

print(len(reseau), "stations")


def voisins(reseau, station):
    return [nom for nom, minutes in reseau[station]]


def temps(reseau, a, b):
    for nom, minutes in reseau[a]:
        if nom == b:
            return minutes
    return None

68 stations

**Q2.1** — Affichez le nombre de stations et le nombre de **liens**.

In [ ]:
# Q2.1
nb_stations = len(reseau)

total_voisins = 0
for station in reseau:
    total_voisins = total_voisins + len(reseau[station])
nb_liens = total_voisins // 2

print("Nombre de stations :", nb_stations)
print("Total des voisins (chaque lien compte 2 fois) :", total_voisins)
print("Nombre de liens :", nb_liens)

Nombre de stations : 68

Total des voisins (chaque lien compté 2 fois) : 138

Nombre de liens : 69

**Observation —** Mon réseau a 68 stations et 69 liens, donc seulement un lien de plus que de stations. Un arbre de 68 stations aurait eu seulement 67 liens. Il y a donc 2 liens en trop, ce qui veut dire que le réseau contient des boucles qui n'est donc pas un arbre.

**Q2.2** — Trouvez la ou les stations qui ont le plus de voisins, puis la liste des
**terminus**.

In [ ]:
# Q2.2
# La station avec le plus de voisins
plus_grand_nb = 0
for station in reseau:
    if len(voisins(reseau, station)) > plus_grand_nb:
        plus_grand_nb = len(voisins(reseau, station))

stations_plus_connectees = []
for station in reseau:
    if len(voisins(reseau, station)) == plus_grand_nb:
        stations_plus_connectees.append(station)

print("Plus grand nombre de voisins :", plus_grand_nb)
print("Station(s) avec ce nombre de voisins :", stations_plus_connectees)

# Les terminus/stations qui n'ont qu'un seul voisin
terminus = []
for station in reseau:
    if len(voisins(reseau, station)) == 1:
        terminus.append(station)

print("Nombre de terminus :", len(terminus))
print("Terminus :", sorted(terminus))

Plus grand nombre de voisins : 5

Station(s) avec ce nombre de voisins : ["Berri-UQAM"]

Nombre de terminus : 6

Terminus : ["Angrignon", "Cote-Vertu", "Honore-Beaugrand", "Longueuil", "Montmorency", "Saint-Michel"]

**Observation —** Berri-UQAM est la seule station avec 5 voisins, le maximum. C'est logique, car c'est là que se croisent les lignes verte, orange et jaune : elle a 2 voisins sur la verte, 2 sur l'orange et 1 sur la jaune. Je trouve 6 terminus (Angrignon, Cote-Vertu, Honore-Beaugrand, Longueuil, Montmorency et Saint-Michel), ce qui correspond aux bouts des lignes.

**Q2.3** — Comptez combien de tronçons durent 1, 2 et 3 minutes.

In [ ]:
# Q2.3
# On compte chaque lien une seule fois (a < b evite de compter 2x)
compte_duree = {1: 0, 2: 0, 3: 0}
for a in reseau:
    for b, minutes in reseau[a]:
        if a < b:
            compte_duree[minutes] = compte_duree[minutes] + 1

for minutes in compte_duree:
    pourcentage = compte_duree[minutes] / nb_liens * 100
    print("Tronçons de", minutes, "minute(s) :", compte_duree[minutes], "(", round(pourcentage), "% )")
print("Total :", sum(compte_duree.values()), "tronçons")

Tronçons de 1 minute(s) : 13 ( 19 % )

Tronçons de 2 minute(s) : 37 ( 54 % )

Tronçons de 3 minute(s) : 19 ( 28 % )

Total : 69 tronçons

> **Observation — obligatoire.** Quelle durée domine, et dans quelle proportion ? Gardez ce chiffre : il servira à la partie 5.
>
> **Observation —** La durée de 2 minutes domine avec 37 tronçons sur 69, soit environ 54 %. Il y a 19 tronçons de 3 minutes (28 %) et 13 de 1 minute (19 %). Je garde ce chiffre comme la majorité des tronçons durent 2 minutes, le nombre d'arrêts et le nombre de minutes devraient souvent donner la même direction.

## Partie 3 — Le parcours en largeur, et son arbre

**Q3.1** — Complétez `bfs(reseau, depart)`. Il rend `distance` (en nombre d'arrêts)
et `precedent`.

In [ ]:
from collections import deque

def bfs(reseau, depart):
    distance = {depart: 0}
    precedent = {}
    file = deque([depart])
    while file:
        station = file.popleft()
        for suivante in voisins(reseau, station):
            if suivante not in distance: # pas encore visitee
                distance[suivante] = distance[station] + 1 # un arret de plus
                precedent[suivante] = station # on vient de station
                file.append(suivante)
    return distance, precedent

> **Observation — obligatoire.** Décrivez en vos mots ce que contient `precedent` à la fin, et à quoi il servira.
>
> **Observation —** À la fin, `precedent` est un dictionnaire qui dit pour chaque station de quelle station on vient pour l'atteindre. Le départ n'y est pas et puisqu'on ne vient de nulle part. Il va servir a retrouver un trajet complet en remontant de l'arrivée jusqu'au départ.

**Q3.2** — Lancez le BFS depuis la station de votre choix. Combien de stations sont
atteintes ? Laquelle est la plus loin, en nombre d'arrêts ?

In [ ]:
# Q3.2
depart_bfs = "Cremazie"
distance_bfs, precedent_bfs = bfs(reseau, depart_bfs)

print("Départ :", depart_bfs)
print("Stations atteintes :", len(distance_bfs), "sur", len(reseau))

# La station la plus loin en nombre d'arrets
plus_loin = depart_bfs
for station in distance_bfs:
    if distance_bfs[station] > distance_bfs[plus_loin]:
        plus_loin = station
print("Station la plus loin :", plus_loin, "à", distance_bfs[plus_loin], "arrêts")

Départ: Cremazie

Stations atteintes: 68 sur 68

Station la plus loin: Angrignon à 21 arrêts

> **Observation — obligatoire.** Le nombre de stations atteintes vous apprend quelque chose sur le réseau : quoi ? Et la station la plus loin vous surprend-elle ?
>
**Observation —** Depuis Cremazie, le BFS atteint 68 stations sur 68, donc toutes les stations. Le réseau est donc d'un seul morceau et on peut aller de n'importe quelle station à n'importe quelle autre. La plus loin est Angrignon, à 21 arrêts, ce qui ne me surprend pas, car elle est au bout de la ligne verte, à l'opposé de Crémazie.

**Q3.3** — Dessinez l'**arbre de parcours** (les 3 premiers niveaux) avec
`graphviz`.

In [ ]:
from graphviz import Digraph

# Q3.3
arbre = Digraph()
arbre.attr(rankdir="TB")

for station in distance_bfs:
    if distance_bfs[station] <= 3:
        arbre.node(station, label=station + "\n(" + str(distance_bfs[station]) + ")")

for station in precedent_bfs:
    if distance_bfs[station] <= 3:
        arbre.edge(precedent_bfs[station], station)

arbre

> **Observation — obligatoire.** Le réseau n'est pas un arbre, et pourtant votre dessin en est un. Expliquez pourquoi.
>
**Observation —** Le réseau a des boucles, mais dans le parcours chaque station n'est ajoutée qu'une seule fois, la première fois qu'on la rencontre. Chaque station (sauf le début) n'a qu'une seule flèche qui arrive vers elle. Les autres liens, qui feraient une boucle, sont ignorés dans le dessin, et c'est pour ça qu'on obtient un arbre.

## Partie 4 — Dijkstra, en minutes

**Q4.1** — Complétez `plus_petite_case`, `relacher` et `dijkstra`.

In [ ]:
INFINI = 9999

def plus_petite_case(distance, barres):
    record = INFINI
    choisie = None
    for station in distance:
        if station not in barres and distance[station] < record:
            record = distance[station]
            choisie = station
    return choisie


def relacher(distance, precedent, a, b, minutes):
    nouvelle = distance[a] + minutes # duree si on passe par a
    if nouvelle < distance[b]: # c'est plus court
        distance[b] = nouvelle
        precedent[b] = a


def dijkstra(reseau, depart):
    distance = {station: INFINI for station in reseau}
    distance[depart] = 0
    precedent = {}
    barres = set()
    while len(barres) < len(reseau):
        station = plus_petite_case(distance, barres)
        if station is None: # stations inatteignables
            break
        barres.add(station)
        for voisine, minutes in reseau[station]:
            relacher(distance, precedent, station, voisine, minutes)
    return distance, precedent

> **Observation — obligatoire.** À quoi sert l'ensemble `barres` ? Que se passerait-il si on l'oubliait ?
>
**Observation —** L'ensemble `barres` contient les stations dont la durée est déjà définitive. Sans lui, `plus_petite_case` choisirait toujours la même station (celle avec la plus petite durée, comme le départ à 0) et le programme tournerait en rond sans jamais avancer. Il sert à passer à une nouvelle station à chaque tour.

**Q4.2** — Complétez `chemin` (la remontée des précédents) et `duree`.

In [ ]:
def chemin(precedent, depart, arrivee):
    route = [arrivee]
    while route[-1] != depart: # on remonte jusqu'au depart
        route.append(precedent[route[-1]])
    route.reverse() # on remet dans le bon sens
    return route


def duree(reseau, route):
    total = 0
    for i in range(len(route) - 1):
        total = total + temps(reseau, route[i], route[i + 1])
    return total

> **Observation — obligatoire.** `chemin` remonte à l'envers puis retourne la liste. Pourquoi ne peut-on pas la construire à l'endroit directement ?
>
**Observation —** On ne peut pas la construire à l'endroit, car `precedent` ne dit que d'où l'on vient, pas où l'on va. On connaît seulement l'arrivée au début, donc on commence par elle et on remonte vers le départ. À la fin, il faut inverser la liste pour la lire dans le bon sens.

**Q4.3** — Affichez le trajet le plus rapide entre deux stations de votre choix,
avec sa durée.

In [ ]:
# Q4.3
depart = "Cremazie"
arrivee = "Angrignon"

distance_d, precedent_d = dijkstra(reseau, depart)
trajet = chemin(precedent_d, depart, arrivee)

print("Trajet le plus rapide de", depart, "à", arrivee, ":")
print(" -> ".join(trajet))
print("Nombre d'arrêts :", len(trajet) - 1)
print("Durée :", duree(reseau, trajet), "minutes")
print("Vérification avec le tableau de Dijkstra :", distance_d[arrivee], "minutes")

Trajet le plus rapide de Cremazie à Angrignon :

Cremazie -> Jarry -> Jean-Talon -> Beaubien -> Rosemont -> Laurier -> Mont-Royal -> Sherbrooke -> 
Berri-UQAM -> Saint-Laurent -> Place-des-Arts -> McGill -> Peel -> Guy-Concordia -> Atwater -> Lionel-Groulx -> 
Charlevoix -> LaSalle -> De l'Eglise -> Verdun -> Jolicoeur -> Monk -> Angrignon

Nombre d'arrêts : 22

Durée : 39 minutes

Vérification avec le tableau de Dijkstra : 39 minutes

> **Observation — obligatoire.** Ce trajet correspond-il à celui que vous prendriez dans la vraie vie ? Si non, qu'est-ce qui manque à notre modèle ?
>
**Observation —** Mon trajet de Cremazie à Angrignon fait 22 arrêts et 39 minutes, en passant par la ligne orange, puis Berri-UQAM, puis la ligne verte. Ça ressemble assez à ce que je prendrais dans la vraie vie, mais notre modèle est simplifié. Il manque le temps d'attente du train, le temps de marche pour une correspondance (ici elles sont gratuites) et le fait que les durées sont seulement approximatives.

## Partie 5 — Comparer les deux, et surligner le chemin

**Q5.1** — Écrivez `comparer(depart, arrivee)` qui affiche, pour chaque algorithme,
le nombre d'arrêts **et** la durée. Testez-la sur **trois** paires de stations, dont
**au moins une** où les deux algorithmes ne donnent pas le même trajet.

In [ ]:
# Q5.1
def comparer(depart, arrivee):
    # Parcours en largeur: le - d'arrets
    dist_b, prec_b = bfs(reseau, depart)
    route_b = chemin(prec_b, depart, arrivee)

    # Dijkstra - de minutes
    dist_d, prec_d = dijkstra(reseau, depart)
    route_d = chemin(prec_d, depart, arrivee)

    print("=====", depart, "->", arrivee, "=====")
    print("BFS      :", len(route_b) - 1, "arrêts,", duree(reseau, route_b), "minutes")
    print("Dijkstra :", len(route_d) - 1, "arrêts,", duree(reseau, route_d), "minutes")
    if route_b == route_d:
        print("Même trajet : oui")
    else:
        print("Même trajet : non")
    print()
    return route_b, route_d


# Trois paires de stations
comparer("Du College", "Honore-Beaugrand")
comparer("Cote-Vertu", "Saint-Michel")
route_bfs_div, route_dij_div = comparer("Cremazie", "Angrignon") #paire divergente

print("Trajet BFS      :", " -> ".join(route_bfs_div))
print("Trajet Dijkstra :", " -> ".join(route_dij_div))

Du College -> Honore-Beaugrand

BFS      : 28 arrêts, 54 minutes

Dijkstra : 28 arrêts, 54 minutes

Même trajet : oui


Cote-Vertu -> Saint-Michel

BFS      : 17 arrêts, 41 minutes

Dijkstra : 17 arrêts, 41 minutes

Même trajet : oui


Cremazie -> Angrignon

BFS      : 21 arrêts, 49 minutes

Dijkstra : 22 arrêts, 39 minutes

Même trajet : non

Trajet BFS      : Cremazie -> Jarry -> Jean-Talon -> De Castelnau -> Parc -> Acadie -> Outremont -> Edouard-Montpetit -> Universite-de-Montreal 
-> Cote-des-Neiges -> Snowdon -> Villa-Maria -> Vendome -> Place-Saint-Henri -> Lionel-Groulx 
-> Charlevoix -> LaSalle -> De l'Eglise -> Verdun -> Jolicoeur -> Monk -> Angrignon

Trajet Dijkstra : Cremazie -> Jarry -> Jean-Talon -> Beaubien -> Rosemont -> Laurier -> Mont-Royal -> Sherbrooke 
-> Berri-UQAM -> Saint-Laurent -> Place-des-Arts -> McGill -> Peel -> Guy-Concordia -> Atwater -> Lionel-Groulx -> Charlevoix 
-> LaSalle -> De l'Eglise -> Verdun -> Jolicoeur -> Monk -> Angrignon



> **Observation — obligatoire.** Sur vos trois paires, à quelle fréquence les deux sont-ils d'accord ? Reliez votre réponse au décompte des durées de la question 2.3.
>
**Observation —** Sur mes trois paires, les deux algorithmes sont d'accord deux fois (Du College à Honore-Beaugrand, et Cote-Vertu à Saint-Michel) et en désaccord une fois (Cremazie à Angrignon). Comme 54 % des tronçons durent 2 minutes, les minutes sont à peu près le double des arrêts, donc les deux algorithmes se ressemblent souvent. Ils diffèrent quand le chemin avec le moins d'arrêts contient beaucoup de tronçons de 3 minutes, comme ici.

**Q5.2** — Sur votre paire divergente, **surlignez le chemin de Dijkstra** dans le
dessin : les arêtes du chemin en **rouge et en épais**, la durée de chaque tronçon en
étiquette, et les stations du chemin remplies d'une couleur.

> **Les attributs à utiliser.** Sur une arête : `color="red"`, `penwidth="3"`,
> `fontcolor="red"`. Sur un sommet : `style="filled"`, `fillcolor="#ffd9d9"`.
> Le reste — quelles arêtes dessiner, comment reconnaître celles du chemin — est à
> vous.

In [ ]:
from graphviz import Graph

# Q5.2
# On met les liens du chemin de Dijkstra dans un ensemble pour les reconnaitre vite.
liens_du_chemin = set()
for i in range(len(route_dij_div) - 1):
    liens_du_chemin.add((route_dij_div[i], route_dij_div[i + 1]))
    liens_du_chemin.add((route_dij_div[i + 1], route_dij_div[i])) # dans les deux sens

dessin = Graph(engine="neato")
dessin.attr(overlap="false", splines="true", size="11,11")
dessin.attr("node", shape="ellipse", fontsize="9")
dessin.attr("edge", fontsize="8", color="gray60")

for station in reseau:
    if station in route_dij_div:
        dessin.node(station, style="filled", fillcolor="#ffd9d9")
    else:
        dessin.node(station)

for a in reseau:
    for b, minutes in reseau[a]:
        if a < b: # chaque lien une seule fois
            if (a, b) in liens_du_chemin:
                dessin.edge(a, b, label=str(minutes), color="red", penwidth="3", fontcolor="red", fontsize="14")
            else:
                dessin.edge(a, b)

dessin

> **Observation — obligatoire.** Décrivez ce que votre dessin rend visible et que la liste de stations ne montrait pas.
>
**Observation —** Le dessin montre où le chemin passe dans le réseau : il traverse le centre par Berri-UQAM, puis Lionel-Groulx, et il évite la ligne bleue. On voit aussi les boucles du réseau, c'est-à-dire qu'il existait un autre chemin possible, ce que la liste de stations ne montrait pas. Les durées en rouge permettent de voir les tronçons de 3 minutes, comme celui entre Lionel-Groulx et Charlevoix.

---
## Partie 6 — Construire un tableau de données

Un modèle a besoin d'un **tableau** : une ligne par exemple, une colonne par
**donnée**, et une colonne à part pour l'**étiquette**.

Ici, **un exemple = une station**, et l'étiquette est sa **ligne**. Les données,
vous les fabriquez avec **votre propre travail des parties 3 et 4**.

On laisse de côté les stations de la ligne jaune : il y en a trop peu.

**Q6.1** — Pour chaque station retenue, fabriquez la liste de ses mesures :

1. son nombre de voisins ;
2. la durée moyenne de ses tronçons ;
3. la durée de son plus long tronçon ;
4. le nombre d'**arrêts** qui la séparent d'un **repère** *(votre BFS)* ;
5. le nombre de **minutes** qui l'en séparent *(votre Dijkstra)*.

Écrivez `mesures(station, nb_reperes)` de façon à pouvoir ajouter des repères
ensuite, et construisez `X_1repere` ainsi que `y`.

In [ ]:
REPERES = ["Berri-UQAM", "Snowdon", "Honore-Beaugrand"]

stations_retenues = [s for s in sorted(reseau) if infos_station[s]["ligne"] != "jaune"]
y = [infos_station[s]["ligne"] for s in stations_retenues]
print(len(stations_retenues), "stations retenues")

# Q6.1
# On calcule une seule fois les distances depuis chaque repere avec BFS et Dijkstra
arrets_depuis = {}
minutes_depuis = {}
for repere in REPERES:
    arrets_depuis[repere] = bfs(reseau, repere)[0]
    minutes_depuis[repere] = dijkstra(reseau, repere)[0]


def mesures(station, nb_reperes):
    # 3 mesures sur la station elle meme
    liste_troncons = reseau[station]
    nb_voisins = len(liste_troncons)
    somme = 0
    plus_long = 0
    for voisine, minutes in liste_troncons:
        somme = somme + minutes
        if minutes > plus_long:
            plus_long = minutes
    moyenne = round(somme / nb_voisins, 2)
    liste = [nb_voisins, moyenne, plus_long]

    # 2 mesures de plus pour chaque repere utilise
    for repere in REPERES[:nb_reperes]:
        liste.append(arrets_depuis[repere][station])
        liste.append(minutes_depuis[repere][station])
    return liste


X_1repere = [mesures(s, 1) for s in stations_retenues]

print("Colonnes : voisins, durée moyenne, durée max, arrêts du repère, minutes du repère")
for i in range(3):
    print(stations_retenues[i], X_1repere[i], "->", y[i])
print("Taille du tableau :", len(X_1repere), "lignes,", len(X_1repere[0]), "colonnes")

66 stations retenues

Colonnes : voisins, durée moyenne, durée max, arrêts du repère, minutes du repère

Acadie [2, 3.0, 3, 9, 17] -> bleue

Angrignon [1, 3.0, 3, 14, 23] -> verte

Assomption [2, 2.5, 3, 8, 16] -> verte

Taille du tableau : 66 lignes, 5 colonnes

> **Observation — obligatoire.** Lesquelles de vos cinq mesures viennent des fichiers fournis, et lesquelles viennent de votre propre code ? Qui a choisi les repères ?
>
**Observation —** Les trois premières mesures (nombre de voisins, durée moyenne et durée maximale) viennent directement des données du fichier CSV. Les deux dernières (arrêts et minutes depuis le repère) viennent de mon propre code, c'est-à-dire mon BFS et mon Dijkstra. Je n'ai pas choisi les repères ils étaient déjà écrits dans le carnet de départ.

**Q6.2** — Comptez combien de stations portent chaque étiquette, et déduisez-en la
note qu'obtiendrait un modèle qui répondrait **toujours la même chose**.

In [ ]:
# Q6.2
nb_par_ligne = {}
for etiquette in y:
    nb_par_ligne[etiquette] = nb_par_ligne.get(etiquette, 0) + 1

for ligne_metro in nb_par_ligne:
    print("Ligne", ligne_metro, ":", nb_par_ligne[ligne_metro], "stations")

# Le modele "toujours la meme chose" repond la ligne qui a le plus de stations
plus_frequente = max(nb_par_ligne, key=nb_par_ligne.get)
note_de_base = nb_par_ligne[plus_frequente] / len(y)
print("Réponse constante :", plus_frequente)
print("Note de ce modèle :", round(note_de_base * 100, 1), "%")

Ligne bleue : 10 stations

Ligne verte : 27 stations

Ligne orange : 29 stations

Réponse constante : orange

Note de ce modèle : 43.9 %

> **Observation — obligatoire.** Notez ce pourcentage. C'est la barre que votre modèle devra dépasser pour servir à quelque chose.
>
**Observation —** Le groupe le plus gros est la ligne orange avec 29 stations sur 66, donc un modèle qui répond toujours « orange » obtient 43,9 %. C'est la barre à dépasser si mon modèle fait moins que ça, il est moins utile que de répondre toujours la même chose.

## Partie 7 — Entraîner un modèle, et le noter honnêtement

**Q7.1** — Coupez en deux paquets (70 / 30), entraînez un `DecisionTreeClassifier`,
et affichez **deux** notes : sur les stations apprises, et sur les stations jamais
vues.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Q7.1
X_train, X_test, y_train, y_test = train_test_split(X_1repere, y, test_size=0.3, random_state=0)

modele = DecisionTreeClassifier(random_state=0)
modele.fit(X_train, y_train)

note_apprises = modele.score(X_train, y_train)
note_jamais_vues = modele.score(X_test, y_test)
print("Stations pour apprendre :", len(X_train), "| stations jamais vues :", len(X_test))
print("Note sur les stations apprises    :", round(note_apprises * 100, 1), "%")
print("Note sur les stations jamais vues :", round(note_jamais_vues * 100, 1), "%")

Stations pour apprendre: 46 | stations jamais vues: 20

Note sur les stations apprises: 95.7 %

Note sur les stations jamais vues: 35.0 %

> **Observation — obligatoire.** Vos deux notes sont très différentes. Comment s'appelle ce phénomène, et laquelle des deux est honnête ?
>
**Observation —** Avec 1 repère, j'obtiens 95,7 % sur les stations apprises mais seulement 35,0 % sur les stations jamais vues. Ce phénomène s'appelle le surapprentissage l'arbre a appris les exemples par cœur. La note honnête est celle des stations jamais vues, et elle est même plus basse que les 43,9 % du modèle qui répond toujours « orange ».

**Q7.2** — Refaites la même chose avec **2** puis **3** repères. Affichez les trois
notes côte à côte.

In [ ]:
# Q7.2
notes_test = {} # on garde la note "jamais vues" de chaque essai

print("Repères | apprises | jamais vues")
for nb in [1, 2, 3]:
    X = [mesures(s, nb) for s in stations_retenues]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
    modele = DecisionTreeClassifier(random_state=0)
    modele.fit(X_train, y_train)
    n_app = modele.score(X_train, y_train)
    n_vues = modele.score(X_test, y_test)
    notes_test[nb] = n_vues
    print("   ", nb, "   |", round(n_app * 100, 1), "%  |", round(n_vues * 100, 1), "%")

Repères | apprises | jamais vues

    1    | 95.7 %  | 35.0 %

    2    | 97.8 %  | 65.0 %
    
    3    | 97.8 %  | 70.0 %

> **Observation — obligatoire.** Qu'est-ce qui change, et pourquoi un seul repère ne suffisait-il pas ? Pensez à deux stations situées de part et d'autre du repère.
>
**Observation —** La note sur les stations jamais vues monte de 35,0 % (1 repère) à 65,0 % (2 repères) puis à 70,0 % (3 repères) alors que la note sur les stations apprises reste autour de 96 à 98 %. Avec un seul repère deux stations de lignes différentes peuvent être à la même distance de Berri-UQAM (une de chaque côté) et avoir les mêmes mesures, donc le modèle ne peut pas les séparer. Avec plus de repères, chaque station a une combinaison de distances plus unique.

**Q7.3** — Sur vos meilleures mesures, comparez un **arbre seul** à une **forêt
aléatoire**. Relancez chacun avec `random_state` valant 0, 1, 2, 3 puis 4, et
affichez pour chaque modèle ses cinq notes, leur **moyenne** et l'**écart** entre la
plus haute et la plus basse.

In [ ]:
# Q7.3
# On prend le nombre de repères qui a donné la meilleure note en Q7.2
meilleur_nb = 1
for nb in notes_test:
    if notes_test[nb] > notes_test[meilleur_nb]:
        meilleur_nb = nb
print("Meilleure mesure :", meilleur_nb, "repère(s)")

X_meilleur = [mesures(s, meilleur_nb) for s in stations_retenues]

for nom_modele in ["Arbre seul", "Forêt aléatoire"]:
    cinq_notes = []
    for graine in [0, 1, 2, 3, 4]:
        X_train, X_test, y_train, y_test = train_test_split(X_meilleur, y, test_size=0.3, random_state=graine)
        if nom_modele == "Arbre seul":
            modele = DecisionTreeClassifier(random_state=graine)
        else:
            modele = RandomForestClassifier(random_state=graine)
        modele.fit(X_train, y_train)
        cinq_notes.append(round(modele.score(X_test, y_test) * 100, 1))

    moyenne = sum(cinq_notes) / len(cinq_notes)
    ecart = max(cinq_notes) - min(cinq_notes)
    print(nom_modele)
    print("   Les 5 notes (%) :", cinq_notes)
    print("   Moyenne :", round(moyenne, 1), "%")
    print("   Écart plus haute - plus basse :", round(ecart, 1), "points")

Meilleure mesure : 3 repère(s)

Arbre seul

   Les 5 notes (%) : [70.0, 65.0, 80.0, 60.0, 60.0]

   Moyenne : 67.0 %

   Écart plus haute - plus basse : 20.0 points

Forêt aléatoire

   Les 5 notes (%) : [65.0, 65.0, 60.0, 60.0, 65.0]

   Moyenne : 63.0 %
   
   Écart plus haute - plus basse : 5.0 points

> **Observation — obligatoire.** La forêt vous a-t-elle aidés ? Appuyez-vous sur la moyenne ET sur l'écart, puis proposez une explication liée au nombre d'exemples dont vous disposez.
>

**Observation —** La forêt ne m'a pas aidé en moyenne: l'arbre seul a 67,0 % de moyenne et la forêt 63,0 %. Par contre, l'écart entre la meilleure et la pire note est de 20 points pour l'arbre et seulement 5 points pour la forêt, donc la forêt est plus stable. Je pense que c'est parce qu'on a très peu d'exemples: 66 stations, donc seulement 20 dans le paquet de test, et chaque station vaut 5 points de pourcentage.

---
## Partie 8 — Vos conclusions

Répondez en Markdown, **en phrases complètes, en français**. Comptez deux ou trois
phrases par question, et **citez vos propres chiffres**.

**Sur le réseau**

1. Le réseau est-il un **arbre** ? Justifiez avec le nombre de stations et de liens.
2. Quelle station est la mieux connectée, et **pourquoi** celle-là ?
3. Sur vos trois paires, le BFS et Dijkstra donnent-ils souvent la même réponse ?
   Reliez votre réponse au décompte des durées.
4. Décrivez **une** paire où ils diffèrent, et expliquez ce que chacun a optimisé.

**Sur le modèle**

5. Avec un seul repère, quelle note obtenez-vous, et comment se compare-t-elle au
   modèle qui répond toujours la même chose ? Qu'est-ce que ça vous apprend ?
6. Qu'est-ce qui change avec 2 puis 3 repères, et **pourquoi** ?
7. Pourquoi la note sur les stations jamais vues est-elle la seule honnête ?
8. La forêt aléatoire vous a-t-elle aidés ? Appuyez-vous sur les cinq notes et sur
   l'écart.

**Sur l'ensemble**

9. Vos données de la partie 6 viennent de vos propres fonctions. Quel risque cela
   pose-t-il si votre BFS ou votre Dijkstra contient une erreur ?
10. Nommez **une** limite de nos données par rapport au vrai métro, et **une**
    mesure supplémentaire que vous auriez aimé avoir pour le modèle.


### Vos réponses

*(remplacez ce texte)*

1. Non, le réseau n'est pas un arbre. Il a 68 stations et 69 liens, alors qu'un arbre de 68 stations aurait seulement 67 liens. Les 2 liens en trop créent des boucles.

2. La station la mieux connectée est Berri-UQAM, avec 5 voisins. C'est parce que trois lignes s'y croisent (verte, orange et jaune), ce qui en fait le centre du réseau.

3. Sur mes trois paires, le BFS et Dijkstra donnent le même trajet deux fois sur trois. Comme 37 tronçons sur 69 (54 %) durent 2 minutes, les minutes sont presque proportionnelles aux arrêts, donc les deux algorithmes se ressemblent souvent.

4. Pour Cremazie à Angrignon, le BFS donne 21 arrêts mais 49 minutes, en passant par la ligne bleue. Dijkstra donne 22 arrêts mais seulement 39 minutes, en passant par Berri-UQAM. Le BFS a optimisé le nombre d'arrêts, et Dijkstra a optimisé le nombre de minutes, donc Dijkstra a un arrêt de plus mais 10 minutes de moins.

5. Avec un seul repère, j'obtiens 35,0 % sur les stations jamais vues, ce qui est moins que les 43,9 % du modèle qui répond toujours « orange ». Ça m'apprend qu'un seul repère ne suffit pas, et que mon modèle était pire que de répondre toujours la même chose.

6. Avec 2 repères la note passe à 65,0 %, puis à 70,0 % avec 3 repères. C'est parce que plusieurs repères donnent plus de mesures différentes, ce qui aide à séparer les stations des trois lignes que 1 seul repère confondait.

7. La note sur les stations apprises est trop belle (95,7 % avec 1 repère), parce que le modèle a déjà vu ces stations et les connaît par cœur. Seules les stations jamais vues (35,0 %) montrent comment le modèle se débrouillera avec de nouvelles stations, donc c'est la seule note honnête.

8. La forêt aléatoire ne m'a pas vraiment aidé sa moyenne est de 63,0 %, contre 67,0 % pour l'arbre seul. Ses cinq notes (65, 65, 60, 60, 65) varient moins que celles de l'arbre (70, 65, 80, 60, 60), avec un écart de 5 points contre 20 points. Elle est donc plus stable, mais pas meilleure, probablement à cause du petit nombre de stations.

9. Si mon BFS ou mon Dijkstra avait une erreur, les deux dernières colonnes de mon tableau seraient fausses sans que je m'en rende compte, et le modèle apprendrait sur de mauvaises données. Ses notes ne montreraient pas le problème, alors il faut bien vérifier les fonctions avant (par exemple, j'ai vérifié que la durée du trajet donne la même chose que le tableau de Dijkstra 39 minutes).

10. Une limite de nos données est que les durées sont approximatives (seulement 1, 2 ou 3 minutes) et qu'il n'y a pas de temps d'attente ni de temps de correspondance, ce qui est différent du vrai métro. Comme mesure supplémentaire, j'aurais aimé avoir la position (latitude et longitude) de chaque station, qui aiderait à séparer les lignes.

---
## Déclaration d'utilisation de l'intelligence artificielle

**Obligatoire, même si vous n'en avez pas utilisé.** Une déclaration absente ou
fausse vaut **0 pour l'ensemble du TP**.

L'IA est permise **pour apprendre**, pas pour produire votre travail. Relisez les
règles de l'énoncé avant de remplir cette section.

**Avez-vous utilisé un outil d'IA ?** _oui / non_ (oui)

**Si oui, remplissez le tableau :**

| Outil | Pour quelle question | Ce que vous lui avez demandé | Lien de partage de la conversation |

Pourquoi if suivante not in distance ?
Pour ne visiter une station qu’une seule fois. Sans ça, le programme tournerait en rond dans les boucles du réseau.

Que fait relacher ?
Elle calcule distance[a] + minutes. Si c’est plus petit que la distance actuelle de b, elle met à jour la distance et note que l’on vient de a.

Pourquoi couper les données en deux paquets (70 / 30) ?
Pour tester le modèle sur des stations qu’il n’a jamais vues. 46 stations servent à apprendre et 20 à tester.

Qu’est-ce que random_state ?
C’est la graine du hasard. Avec la même valeur, on obtient la même coupe des données et le même arbre. En essayant 0 à 4, on voit à quel point la note dépend de la chance.

Demande aussi pour les fautes d'orthographe.


**Confirmez, en écrivant « oui » :**

- Je n'ai **pas** collé l'énoncé du TP, en tout ou en partie, dans une conversation
  avec une IA. → __oui___
- Je n'ai **pas** collé le contenu des fichiers de données. → __oui___
- Je comprends chaque ligne de code que je remets et je peux l'expliquer à l'oral.
  → ___oui__

---
**Qui a fait quoi :** *(obligatoire si vous êtes deux)*

---
## Avant de remettre

**Vingt points du TP ne portent pas sur vos réponses** : dix sur la tenue de ce
carnet, dix sur votre usage de Git. Relisez les sections 3 et 4 de l'énoncé avant de
déposer votre travail.

**Le carnet.** Faites **Kernel → Redémarrer et tout exécuter** en dernier, et
vérifiez qu'aucune cellule ne produit d'erreur. Puis relisez-le comme si vous ne
l'aviez pas écrit : est-ce qu'on s'y retrouve ? Est-ce que chaque nombre affiché
s'explique sans aller lire le code ? Reste-t-il des cellules d'essai, des variables
au nom obscur, des observations expédiées ? Le français est-il relu ?

**Le dépôt.** Le nom de votre branche et vos messages de commit suivent la
convention vue en cours. Votre historique doit montrer un travail construit au fil du
temps — et, si vous êtes deux, la contribution des deux.

**L'identification.** Nom, coéquipier et groupe en première cellule, « qui a fait
quoi » à la fin, déclaration d'IA remplie même si la réponse est « non », et le
dossier `donnees/` à côté du carnet.

---

*Ensuite : `Publish branch`, `Create Pull Request`, fusion dans `main`, et le lien du
dépôt sur Omnivox.*